Feature engineering!

the goal of feature engineering is to turn book information into numerical features that a computer can compare. The dataset currently describes books using columns such as author, genres, page_count, average_rating, and description. A recommendation algorithm needs a numerical representation of those characteristics to calculate similarity.

In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
# load in the dataset

df = pd.read_csv("../data/processed/books_clean.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 99
Columns: 9


,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description
0,B001,Dune,Frank Herbert,Science Fiction;Politics;Adventure,1965,412,4.27,850000,A young nobleman becomes entangled in a strugg...
1,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,1989,482,4.22,185000,Seven travelers journey toward a mysterious wo...
2,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,1969,304,4.08,190000,An envoy visits a distant planet and must navi...
3,B004,The Dispossessed,Ursula K. Le Guin,Science Fiction;Politics;Literary,1974,387,4.23,170000,A physicist moves between contrasting worlds w...
4,B005,Foundation,Isaac Asimov,Science Fiction;Space Opera;Politics,1951,296,4.17,500000,A mathematician predicts the collapse of a vas...


Let's take a look specifically at the columns we are using for this stage.

We'll focus on two types of features initially:
- Genres: the book's categories, such as Fantasy, Adventure, or Science Fiction.
- Description: words describing the book's plot, setting, and themes.

In [3]:
df[["book_id", "title", "author", "genres", "description"]].head()

,book_id,title,author,genres,description
0,B001,Dune,Frank Herbert,Science Fiction;Politics;Adventure,A young nobleman becomes entangled in a strugg...
1,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,Seven travelers journey toward a mysterious wo...
2,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,An envoy visits a distant planet and must navi...
3,B004,The Dispossessed,Ursula K. Le Guin,Science Fiction;Politics;Literary,A physicist moves between contrasting worlds w...
4,B005,Foundation,Isaac Asimov,Science Fiction;Space Opera;Politics,A mathematician predicts the collapse of a vas...


Let's focus on the genres. They are stored as semicolon separated text.

In [5]:
#splits genres into lists
genre_lists = df["genres"].fillna("").apply(
    lambda value: value.split(";") if value else []
)

#fillna("") handles missing genre values by treating them as an empty string before splitting

genre_lists.head(10)

0       [Science Fiction, Politics, Adventure]
1    [Science Fiction, Space Opera, Adventure]
2        [Science Fiction, Politics, Literary]
3        [Science Fiction, Politics, Literary]
4     [Science Fiction, Space Opera, Politics]
5       [Science Fiction, Survival, Adventure]
6          [Science Fiction, Space, Adventure]
7          [Science Fiction, Cyberpunk, Crime]
8      [Science Fiction, Cyberpunk, Adventure]
9          [Science Fiction, Physics, Mystery]
Name: genres, dtype: object

Now we convert genres into numerical features. Each unique genre becomes a column, and each book gets a 1 or 0 depending on whether it belongs to that genre.

In [6]:
mlb = MultiLabelBinarizer()

genre_features = mlb.fit_transform(genre_lists)

genre_df = pd.DataFrame(
    genre_features,
    columns=mlb.classes_,
    index=df.index
)

genre_df.head()

,Adventure,Anthropology,Apocalypse,Arthurian,Biography,Business,Celebrity,Character,Classic,Climate,...,Space,Space Opera,Speculative,Survival,Technology,Thriller,Time Travel,War,Western,Young Adult
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [7]:
#let's check the genre features

print("Number of unique genres:", len(mlb.classes_))
print("Genre feature shape:", genre_df.shape)

print("Genre columns:")
print(genre_df.columns.tolist())

Number of unique genres: 68
Genre feature shape: (99, 68)
Genre columns:
['Adventure', 'Anthropology', 'Apocalypse', 'Arthurian', 'Biography', 'Business', 'Celebrity', 'Character', 'Classic', 'Climate', 'Comedy', 'Coming of Age', 'Contemporary', 'Crime', 'Cyberpunk', 'Design', 'Dragons', 'Drama', 'Dystopian', 'Ecology', 'Economics', 'Education', 'Epic', 'Family', 'Fantasy', 'Friendship', 'Gothic', 'Historical', 'History', 'Immigration', 'Investigative', 'Justice', 'Literary', 'Magic', 'Magical Realism', 'Mars', 'Medical History', 'Memoir', 'Migration', 'Mystery', 'Mythology', 'Nature', 'Nonfiction', 'Philosophical', 'Physics', 'Politics', 'Post-Apocalyptic', 'Productivity', 'Psychological', 'Psychology', 'Rebellion', 'Revenge', 'Romance', 'Satire', 'Science', 'Science Fiction', 'Self Help', 'Society', 'Space', 'Space Opera', 'Speculative', 'Survival', 'Technology', 'Thriller', 'Time Travel', 'War', 'Western', 'Young Adult']


Genres are useful, but books can be the same genre and still vary wildly. Let's get into the description. We'll convert description text into word-base features.

In [8]:
#first let's inspect them
df["description"].head(10)


0    A young nobleman becomes entangled in a strugg...
1    Seven travelers journey toward a mysterious wo...
2    An envoy visits a distant planet and must navi...
3    A physicist moves between contrasting worlds w...
4    A mathematician predicts the collapse of a vas...
5    An astronaut stranded on Mars uses engineering...
6    An astronaut awakens alone on a desperate miss...
7    A damaged computer hacker is hired for a dange...
8    A hacker and courier investigate a mysterious ...
9    Scientists investigating strange phenomena unc...
Name: description, dtype: str

We'll do this by creating a sparse matrix, a memory-efficient format that stores mostly nonzero values without filling memory with zeros.

In [10]:
vectorizer = CountVectorizer(
    stop_words="english", #removes common English words, like “the” and “and,”
    binary=True #yes/no for if a word appears, does not count number of appearances
)

description_features = vectorizer.fit_transform(
    df["description"].fillna("")
)

Let's examine the description features

In [11]:
print("Description feature shape:", description_features.shape)

print("First 20 words:")
print(vectorizer.get_feature_names_out()[:20])

Description feature shape: (99, 708)
First 20 words:
['1950s' 'abandons' 'abbey' 'abilities' 'ability' 'academy' 'account'
 'accused' 'achilles' 'acquires' 'actions' 'administrative' 'adulthood'
 'affect' 'affluent' 'age' 'aging' 'alaskan' 'ally' 'alternate']


In [19]:
#inspect one book's description features

book_index = 1

book_words = pd.Series(
    description_features[book_index].toarray()[0],
    index=vectorizer.get_feature_names_out()
)

book_words[book_words > 0].sort_values(ascending=False)

brought        1
experiences    1
journey        1
mysterious     1
recounting     1
seven          1
travelers      1
world          1
dtype: int64

At this point, you have two separate numerical representations:
- genre_df: genre indicators, with one column per genre.
- description_features: word indicators, with one column per vocabulary word.

The code below combines the genre indicators and description word indicators side by side.
For example, if you had 25 genre columns and 1,500 description-word columns, the combined matrix would have 1,525 features per book.

In [21]:
from scipy.sparse import hstack, csr_matrix

combined_features = hstack([
    csr_matrix(genre_df.to_numpy()),
    description_features
]).tocsr()

print("Combined feature shape:", combined_features.shape)

Combined feature shape: (99, 776)


In [ ]:
#now let's save the genre features

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True) #creates processed folder if not already there

genre_output = pd.concat(
    [df[["book_id", "title"]].reset_index(drop=True),
     genre_df.reset_index(drop=True)],
    axis=1
)

genre_output.to_csv(
    output_dir / "book_genre_features.csv",
    index=False
)

In [24]:
#saving description features

from scipy.sparse import save_npz

save_npz(
    output_dir / "book_description_features.npz", #description matrix is sparse, so we'll save it in a format designed for sparse matrices
    description_features
)

In [25]:
#saving the combined features
save_npz(
    output_dir / "book_combined_features.npz",
    combined_features
)

In [26]:
#let's verify that they all saved correctly; all 3 should return true

print("Genre feature file exists:",
      (output_dir / "book_genre_features.csv").exists())

print("Description feature file exists:",
      (output_dir / "book_description_features.npz").exists())

print("Combined feature file exists:",
      (output_dir / "book_combined_features.npz").exists())

Genre feature file exists: True
Description feature file exists: True
Combined feature file exists: True
